In [155]:
import yfinance as yf
import pandas as pd
import subprocess
import sys

In [156]:
print(f"Current version: {yf.__version__}")

Current version: 1.3.0


In [ ]:
ticker_list = ['2267.T', '3435.T', '1846.HK', '6889.HK', '7399.T', '1913.HK', 'VTU.L']

import pandas as pd
import yfinance as yf


def pull_yf_ticker_data(ticker_list):
    extracted_data = []

    for ticker_str in ticker_list:
        print(f"Fetching data for: {ticker_str}...")
        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            # 1. Fetch currencies
            trading_curr = info.get("currency", None).upper()
            financial_curr = info.get("financialCurrency", None).upper()
            market_cap = info.get("marketCap", None)

            # 2. Convert Market Cap if currencies don't match
            market_cap_converted = market_cap

            if (
                trading_curr
                and financial_curr
                and trading_curr != financial_curr
            ):
                fx_ticker_str = f"{trading_curr}{financial_curr}=X"
                print(
                    f"   ⚠️ Currency mismatch detected ({trading_curr} vs {financial_curr}). Fetching FX rate {fx_ticker_str}..."
                )

                try:
                    fx_ticker = yf.Ticker(fx_ticker_str)
                    # Get the most recent closing price for the currency pair
                    fx_data = fx_ticker.history(period="1d")
                    if not fx_data.empty:
                        fx_rate = fx_data["Close"].iloc[-1]
                        if market_cap:
                            market_cap_converted = market_cap * fx_rate
                            print(
                                f"   Converted Market Cap using rate: {fx_rate:.4f}"
                            )
                    else:
                        print(
                            f"   Could not fetch FX data for {fx_ticker_str}. Using original Market Cap."
                        )
                except Exception as fx_error:
                    print(f"   Error fetching FX conversion rate: {fx_error}")

            # 3. Balance Sheet Data
            q_bs = ticker.quarterly_balance_sheet
            if not q_bs.empty:
                latest_date = q_bs.columns[0]
                latest_col = q_bs[latest_date]

                total_assets = latest_col.get("Total Assets", None)
                current_assets = latest_col.get("Current Assets", None)
                total_liabilities = latest_col.get(
                    "Total Liabilities Net Minority Interest", None
                )
                total_investments = latest_col.get("Investment Properties", None)
                total_goodwill_intangibles = latest_col.get("Goodwill And Other Intangible Assets", None)
            else:
                total_assets = (
                    current_assets
                ) = total_liabilities = total_investments = None

            # --- Extract Latest FY Net Income ---
            annual_inc = ticker.income_stmt
            latest_fy_net_inc = (
                annual_inc.loc["Net Income"].iloc[0]
                if not annual_inc.empty and "Net Income" in annual_inc.index
                else None
            )

            # --- Extract TTM Net Income ---
            ttm_inc = ticker.get_income_stmt(modes="trailing")
            if not ttm_inc.empty and "Net Income" in ttm_inc.index:
                ttm_net_inc = ttm_inc.loc["Net Income"].iloc[0]
            else:
                # Fallback calculation
                q_inc = ticker.quarterly_income_stmt
                ttm_net_inc = (
                    q_inc.loc["Net Income"].iloc[:4].sum()
                    if not q_inc.empty and "Net Income" in q_inc.index
                    else None
                )

            # Append everything to our dataset
            extracted_data.append(
                {
                    "Ticker": ticker_str,
                    "Trading Currency": trading_curr,
                    "Financial Currency": financial_curr,
                    # We pass the converted Market Cap forward to keep ratios accurate
                    "Market Cap": market_cap_converted,
                    "Total Assets": total_assets,
                    "Total Current Assets": current_assets,
                    "Total Goodwill and Intangibles": total_goodwill_intangibles,
                    "Total Liabilities": total_liabilities,
                    "Total Investments": total_investments,
                    "Latest FY Net Income": latest_fy_net_inc,
                    "TTM Net Income": ttm_net_inc,
                }
            )

        except Exception as e:
            print(f"Error fetching data for {ticker_str}: {e}")
            extracted_data.append(
                {
                    "Ticker": ticker_str,
                    "Trading Currency": None,
                    "Financial Currency": None,
                    "Market Cap": None,
                    "Total Assets": None,
                    "Total Current Assets": None,
                    "Total Goodwill and Intangibles": None,
                    "Total Liabilities": None,
                    "Total Investments": None,
                    "Latest FY Net Income": None,
                    "TTM Net Income": None,
                }
            )

    return pd.DataFrame(extracted_data)

In [158]:
def format_currency(val):
    """Formats large numbers into human-readable M (Millions) and B (Billions) strings."""
    # Handle absolute values for the logic, but preserve the negative sign
    abs_val = abs(val)

    if abs_val >= 1_000_000_000_000:
        return f"{val / 1_000_000_000_000:.1f}t"
    elif abs_val >= 1_000_000_000:
        return f"{val / 1_000_000_000:.1f}b"
    elif abs_val >= 1_000_000:
        return f"{val / 1_000_000:.1f}m"
    elif abs_val >= 1_000:
        return f"{val / 1_000:.1f}k"

    return (
        str(int(val)) if val == int(val) else f"{val:.1f}"
    )  # Return as string if small

def format_ratio(val):
    """Returns 'N/A' if the ratio is negative, infinite, or NaN, otherwise formats to 2 decimals."""
    # Check for negative values, NaN (from division by zero), or infinity
    if pd.isna(val) or val < 0 or val == float("inf") or val == float("-inf"):
        return "N/A"
    return f"{val:.2f}"

In [159]:
df = pull_yf_ticker_data(ticker_list=ticker_list)

# Your updated pipeline
formatted_df = (
    df.fillna(0)
    .assign(
        NCAV=lambda df: (df["Total Current Assets"] - df["Total Liabilities"]),
        NCAV_Inv=lambda df: (
            df["Total Current Assets"]
            - df["Total Liabilities"]
            + df["Total Investments"]
        ),
        Book_Value=lambda df: (df["Total Assets"] - df["Total Liabilities"]),
        Tangible_Book_Value=lambda df: (df["Total Assets"] - df["Total Liabilities"] - df["Total Goodwill and Intangibles"]),
        Price_Book_Ratio=lambda df: (df["Market Cap"] / df["Book_Value"]),
        Price_Tangible_Book_Ratio=lambda df: (df["Market Cap"] / df["Tangible_Book_Value"]),
        Price_NCAV_Ratio=lambda df: (df["Market Cap"] / df["NCAV"]),
        Price_NCAV_Inv_Ratio=lambda df: (df["Market Cap"] / df["NCAV_Inv"])
    )
    .reindex()
)  # Best practice before applying string formatting

# Apply the formatting function to the specific financial columns
columns_to_format = [
    "Market Cap",
    "Total Assets",
    "Total Current Assets",
    "Total Liabilities",
    "Total Investments",
    "Total Goodwill and Intangibles",
    "NCAV",
    "NCAV_Inv",
    "Book_Value",
    "Tangible_Book_Value",
    "Latest FY Net Income",
    "TTM Net Income"
]

for col in columns_to_format:
    if col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].map(format_currency)

# 2. Apply ratio formatting (converting negative/inf ratios to "N/A")
columns_to_format_ratios = [
    "Price_Book_Ratio",
    "Price_Tangible_Book_Ratio",
    "Price_NCAV_Ratio",
    "Price_NCAV_Inv_Ratio",
]

for col in columns_to_format_ratios:
    if col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].map(format_ratio)

formatted_df

Fetching data for: 2267.T...
Error fetching data for 2267.T: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'
Fetching data for: 3435.T...
Error fetching data for 3435.T: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'
Fetching data for: 1846.HK...
Error fetching data for 1846.HK: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'
Fetching data for: 6889.HK...
   ⚠️ Currency mismatch detected (HKD vs JPY). Fetching FX rate HKDJPY=X...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


   Converted Market Cap using rate: 20.2745
Error fetching data for 6889.HK: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'
Fetching data for: 7399.T...
Error fetching data for 7399.T: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'
Fetching data for: 1913.HK...
   ⚠️ Currency mismatch detected (HKD vs EUR). Fetching FX rate HKDEUR=X...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


   Converted Market Cap using rate: 0.1098
Error fetching data for 1913.HK: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'
Fetching data for: VTU.L...
Error fetching data for VTU.L: TickerBase.get_income_stmt() got an unexpected keyword argument 'modes'


,Ticker,Trading Currency,Financial Currency,Market Cap,Total Assets,Total Current Assets,Total Goodwill and Intangibles,Total Liabilities,Total Investments,Latest FY Net Income,TTM Net Income,NCAV,NCAV_Inv,Book_Value,Tangible_Book_Value,Price_Book_Ratio,Price_Tangible_Book_Ratio,Price_NCAV_Ratio,Price_NCAV_Inv_Ratio
0,2267.T,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
1,3435.T,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
2,1846.HK,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
3,6889.HK,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
4,7399.T,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
5,1913.HK,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
6,VTU.L,0,0,0,0,0,0,0,0,0,0,0,0,0,0,N/A,N/A,N/A,N/A
